In [1]:
from pathlib import Path
print("CWD =", Path.cwd())
print("DATA_DIR exists =", Path("data").exists())
print("Files in data =", list(Path("data").glob("*.csv")))

CWD = /Users/lejlakacaeva/Desktop/aie-leyla-2025/homeworks/HW07
DATA_DIR exists = True
Files in data = [PosixPath('data/S07-hw-dataset-02.csv'), PosixPath('data/S07-hw-dataset-03.csv'), PosixPath('data/S07-hw-dataset-04.csv')]


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

In [3]:
DATA_DIR = Path("data")
ART_DIR = Path("artifacts")
FIG_DIR = ART_DIR / "figures"
LAB_DIR = ART_DIR / "labels"

FIG_DIR.mkdir(parents=True, exist_ok=True)
LAB_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = [
    "S07-hw-dataset-02.csv",
    "S07-hw-dataset-03.csv",
    "S07-hw-dataset-04.csv",
]

In [4]:
def evaluate_clustering(X, labels):
    labels = np.asarray(labels)

    # если 0/1 кластер или все точки шум (-1), метрики не определены
    uniq = np.unique(labels)
    if len(uniq) < 2:
        return {
            "n_clusters": int(len(uniq)),
            "silhouette": None,
            "davies_bouldin": None,
            "calinski_harabasz": None,
            "noise_share": float(np.mean(labels == -1)),
        }

    # Для DBSCAN часто считают метрики на non-noise точках
    mask = labels != -1
    noise_share = float(np.mean(~mask))

    if mask.sum() >= 2 and len(np.unique(labels[mask])) >= 2:
        X_eval = X[mask]
        y_eval = labels[mask]
        sil = float(silhouette_score(X_eval, y_eval))
        db  = float(davies_bouldin_score(X_eval, y_eval))
        ch  = float(calinski_harabasz_score(X_eval, y_eval))
        ncl = int(len(np.unique(y_eval)))
    else:
        sil = db = ch = None
        ncl = int(len(np.unique(labels[mask])))

    return {
        "n_clusters": ncl,
        "silhouette": sil,
        "davies_bouldin": db,
        "calinski_harabasz": ch,
        "noise_share": noise_share,
    }


def pca_scatter(X, labels, title, out_path):
    pca = PCA(n_components=2, random_state=42)
    X2 = pca.fit_transform(X)

    plt.figure(figsize=(6, 5))
    plt.scatter(X2[:, 0], X2[:, 1], c=labels, s=10, cmap="tab10")
    plt.title(title)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.tight_layout()
    plt.savefig(out_path, dpi=160)
    plt.close()

In [5]:
def run_one_dataset(filename):
    path = DATA_DIR / filename
    df = pd.read_csv(path)

    print("====", filename, "====")
    display(df.head())
    display(df.info())
    display(df.describe(include="all").T.head(10))
    print("Missing values share (top):")
    display((df.isna().mean().sort_values(ascending=False).head(10)))

    sample_id = df["sample_id"].copy()
    X_raw = df.drop(columns=["sample_id"])

    # разделяем типы колонок
    cat_cols = X_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num_cols = [c for c in X_raw.columns if c not in cat_cols]

    # препроцессинг: числовые -> imputer + scaler, категориальные -> imputer + onehot
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
    )

    # для метрик и PCA удобно получить матрицу признаков
    X = preprocessor.fit_transform(X_raw)

    # X может быть sparse из-за onehot -> переведём в dense для PCA/DBSCAN на небольших данных
    if hasattr(X, "toarray"):
        X = X.toarray()

    results = []
    best = None  # (score, info)

    # ---- KMeans: подбор k ----
    ks = list(range(2, 21))
    km_sil = []

    for k in ks:
        km = KMeans(n_clusters=k, n_init=20, random_state=42)  # n_init фиксируем для стабильности [web:316]
        labels = km.fit_predict(X)
        m = evaluate_clustering(X, labels)
        m.update({"dataset": filename, "algo": "kmeans", "k": k})
        results.append(m)
        km_sil.append(m["silhouette"] if m["silhouette"] is not None else np.nan)

        # выбираем лучшего по silhouette (можно поменять правило позже)
        score = m["silhouette"]
        if score is not None and (best is None or score > best[0]):
            best = (score, {"algo": "kmeans", "params": {"k": k}, "labels": labels})

    # график silhouette vs k
    plt.figure(figsize=(6, 4))
    plt.plot(ks, km_sil, marker="o")
    plt.title(f"{filename}: KMeans silhouette vs k")
    plt.xlabel("k")
    plt.ylabel("silhouette")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{filename.replace('.csv','')}_kmeans_sil_vs_k.png", dpi=160)
    plt.close()

    # ---- DBSCAN: подбор eps/min_samples (грубая сетка) ----
    # Важно: eps подбирается в масштабе "после scaler/onehot" [web:290]
    eps_grid = [0.3, 0.5, 0.7, 1.0, 1.5]
    min_samples_grid = [5, 10, 20]

    db_best_local = None  # (sil, eps, ms, labels, noise_share)
    for eps in eps_grid:
        for ms in min_samples_grid:
            dbs = DBSCAN(eps=eps, min_samples=ms)
            labels = dbs.fit_predict(X)
            m = evaluate_clustering(X, labels)
            m.update({"dataset": filename, "algo": "dbscan", "eps": eps, "min_samples": ms})
            results.append(m)

            # выбираем лучшего DBSCAN по silhouette (на non-noise)
            score = m["silhouette"]
            if score is not None and (db_best_local is None or score > db_best_local[0]):
                db_best_local = (score, eps, ms, labels, m["noise_share"])

            # общий best тоже обновляем
            if score is not None and (best is None or score > best[0]):
                best = (score, {"algo": "dbscan", "params": {"eps": eps, "min_samples": ms}, "labels": labels})

    # график: silhouette vs eps для лучшего min_samples (если нашлось)
    if db_best_local is not None:
        best_ms = db_best_local[2]
        xs, ys = [], []
        for eps in eps_grid:
            row = [r for r in results if r["algo"]=="dbscan" and r["min_samples"]==best_ms and r["eps"]==eps][0]
            xs.append(eps)
            ys.append(row["silhouette"] if row["silhouette"] is not None else np.nan)

        plt.figure(figsize=(6, 4))
        plt.plot(xs, ys, marker="o")
        plt.title(f"{filename}: DBSCAN silhouette vs eps (min_samples={best_ms})")
        plt.xlabel("eps")
        plt.ylabel("silhouette (non-noise)")
        plt.tight_layout()
        plt.savefig(FIG_DIR / f"{filename.replace('.csv','')}_dbscan_sil_vs_eps.png", dpi=160)
        plt.close()

    # ---- PCA scatter для лучшего решения ----
    if best is not None:
        best_info = best[1]
        labels_best = best_info["labels"]
        title = f"{filename}: best={best_info['algo']} params={best_info['params']} (sil={best[0]:.3f})"
        pca_scatter(X, labels_best, title, FIG_DIR / f"{filename.replace('.csv','')}_pca_best.png")

        # сохранить labels
        out_labels = pd.DataFrame({"sample_id": sample_id, "cluster_label": labels_best})
        out_labels.to_csv(LAB_DIR / f"labels_{filename.replace('.csv','')}.csv", index=False)

    # вернуть всё для общих json
    return results, best

In [6]:
all_rows = []
best_configs = {}

for fn in DATASETS:
    rows, best = run_one_dataset(fn)
    all_rows.extend(rows)

    if best is None:
        best_configs[fn] = None
    else:
        best_configs[fn] = {
            "criterion": "silhouette (non-noise for DBSCAN)",
            "best_score": float(best[0]),
            "algo": best[1]["algo"],
            "params": best[1]["params"],
        }

metrics_df = pd.DataFrame(all_rows)
display(metrics_df.head())

# сохранить артефакты
metrics_summary = metrics_df.to_dict(orient="records")
(ART_DIR / "metrics_summary.json").write_text(json.dumps(metrics_summary, ensure_ascii=False, indent=2))

(ART_DIR / "best_configs.json").write_text(json.dumps(best_configs, ensure_ascii=False, indent=2))

==== S07-hw-dataset-02.csv ====


,sample_id,x1,x2,z_noise
0,0,0.098849,-1.846034,21.288122
1,1,-1.024516,1.829616,6.072952
2,2,-1.094178,-0.158545,-18.938342
3,3,-1.612808,-1.565844,-11.629462
4,4,1.659901,-2.133292,1.895472


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  8000 non-null   int64  
 1   x1         8000 non-null   float64
 2   x2         8000 non-null   float64
 3   z_noise    8000 non-null   float64
dtypes: float64(3), int64(1)
memory usage: 250.1 KB


None

,count,mean,std,min,25%,50%,75%,max
sample_id,8000.0,3999.500000,2309.545410,0.000000,1999.750000,3999.500000,5999.250000,7999.000000
x1,8000.0,0.478867,0.955138,-2.487352,-0.116516,0.490658,1.085263,2.987555
x2,8000.0,0.241112,0.663195,-2.499237,-0.242357,0.241092,0.726526,2.995553
z_noise,8000.0,0.110454,8.097716,-34.056074,-5.392210,0.132470,5.655605,29.460076


...

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
sample_id,10000.0,NaN,NaN,NaN,4999.5,2886.89568,0.0,2499.75,4999.5,7499.25,9999.0
cat_a,10000,6,E,1692,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cat_b,10000,6,V,1682,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n01,9826.0,NaN,NaN,NaN,17.348435,22.578551,-22.43709,-3.975438,22.042807,37.535647,65.446912
n02,9811.0,NaN,NaN,NaN,-2.05762,19.04341,-37.546998,-14.200552,-6.532183,2.092197,43.326647
n03,9801.0,NaN,NaN,NaN,7.908302,25.637807,-38.136412,-8.591513,0.3504,30.72563,60.185729
n04,9808.0,NaN,NaN,NaN,14.269157,18.815319,-23.374316,-1.223379,10.069142,29.807101,65.094588
n05,9799.0,NaN,NaN,NaN,0.90059,20.981294,-45.91407,-5.086756,2.413111,18.398883,42.527554
n06,9817.0,NaN,NaN,NaN,5.832787,13.221646,-20.650038,-4.532057,7.391953,13.033076,39.933274
n07,9796.0,NaN,NaN,NaN,-0.840875,26.583849,-60.297304,-13.55472,-2.429024,16.095731,48.591236


Missing values share (top):


n26    0.0224
n21    0.0215
n18    0.0212
n17    0.0212
n28    0.0211
n24    0.0207
n07    0.0204
n11    0.0204
n20    0.0203
n29    0.0202
dtype: float64

,n_clusters,silhouette,davies_bouldin,calinski_harabasz,noise_share,dataset,algo,k,eps,min_samples
0,2,0.306861,1.323472,3573.393333,0.0,S07-hw-dataset-02.csv,kmeans,2.0,NaN,NaN
1,3,0.270084,1.225312,3082.774289,0.0,S07-hw-dataset-02.csv,kmeans,3.0,NaN,NaN
2,4,0.251262,1.300148,2915.587770,0.0,S07-hw-dataset-02.csv,kmeans,4.0,NaN,NaN
3,5,0.252392,1.217310,2704.355372,0.0,S07-hw-dataset-02.csv,kmeans,5.0,NaN,NaN
4,6,0.259820,1.159521,2571.090931,0.0,S07-hw-dataset-02.csv,kmeans,6.0,NaN,NaN


640

In [7]:
from sklearn.metrics import adjusted_rand_score

fn = "S07-hw-dataset-04.csv"
df = pd.read_csv(DATA_DIR / fn)
sample_id = df["sample_id"].copy()
X_raw = df.drop(columns=["sample_id"])

cat_cols = X_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in X_raw.columns if c not in cat_cols]

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])

preprocessor = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)], remainder="drop")

X = preprocessor.fit_transform(X_raw)
if hasattr(X, "toarray"):
    X = X.toarray()

k = best_configs[fn]["params"]["k"] if best_configs[fn] and best_configs[fn]["algo"]=="kmeans" else 8

labels_runs = []
seeds = [0, 1, 2, 3, 4]
for s in seeds:
    km = KMeans(n_clusters=k, n_init=20, random_state=s)
    labels_runs.append(km.fit_predict(X))

# матрица ARI
ari = np.zeros((len(seeds), len(seeds)))
for i in range(len(seeds)):
    for j in range(len(seeds)):
        ari[i, j] = adjusted_rand_score(labels_runs[i], labels_runs[j])

ari_df = pd.DataFrame(ari, index=[f"seed={s}" for s in seeds], columns=[f"seed={s}" for s in seeds])
display(ari_df)

ari_df.to_csv(ART_DIR / "stability_ari_kmeans_ds4.csv", index=True)

,seed=0,seed=1,seed=2,seed=3,seed=4
seed=0,1.000000,0.923802,0.862404,0.924586,0.923295
seed=1,0.923802,1.000000,0.872722,0.991902,0.986010
seed=2,0.862404,0.872722,1.000000,0.872237,0.880400
seed=3,0.924586,0.991902,0.872237,1.000000,0.982601
seed=4,0.923295,0.986010,0.880400,0.982601,1.000000
